# Part 1: Collecting the Data

For this project I needed monthly trade data for smartphones. I used the **UN Comtrade** database, which collects
the import and export figures that every country's customs department reports to the United Nations.

Comtrade has a free *preview* API that works without an account. The catch is that it only gives **one month per
request** (max 500 rows), so I have to loop over every month and save each one.

**What I am downloading**

| Dataset | Reporter | Flow | Meaning |
|---|---|---|---|
| US imports | 842 (USA) | M | Smartphones the USA bought from each country |
| India exports | 699 (India) | X | Smartphones India sold to each country |

Product: **HS code 851713 = "Smartphones"**. This code was only created in the HS 2022 update. Before 2022,
smartphones were mixed together with ordinary mobile phones, so my data starts from **January 2022**.

I save every API response as a raw JSON file without changing anything. All the cleaning happens in the next notebook,
so I can always go back and check the original data.

In [1]:
import requests
import json
import os
import time

In [2]:
BASE_URL = "https://comtradeapi.un.org/public/v1/preview/C/M/HS"
HS_CODE = "851713"   # smartphones

US_FOLDER = "../data/raw/us_imports"
INDIA_FOLDER = "../data/raw/india_exports"
os.makedirs(US_FOLDER, exist_ok=True)
os.makedirs(INDIA_FOLDER, exist_ok=True)

# all months from Jan 2022 to Aug 2026 (the latest months may not be published yet)
months = []
for year in range(2022, 2027):
    for month in range(1, 13):
        if year == 2026 and month > 8:
            break
        months.append(f"{year}{month:02d}")

print("Total months to try:", len(months))
print(months[:3], "...", months[-3:])

Total months to try: 56
['202201', '202202', '202203'] ... ['202606', '202607', '202608']


### Function to download one month

If a file is already saved I skip the download, so re-running the notebook doesn't call the API again.
If the API says "too many requests" (status 429) I wait and try again.

In [3]:
def download_month(reporter, flow, period, folder):
    file_path = f"{folder}/{period[:4]}-{period[4:]}.json"

    # already downloaded before
    if os.path.exists(file_path):
        with open(file_path) as f:
            return len(json.load(f)["data"])

    params = {"reporterCode": reporter, "period": period, "cmdCode": HS_CODE, "flowCode": flow}

    for attempt in range(3):
        response = requests.get(BASE_URL, params=params, timeout=60)
        if response.status_code == 429:
            print("  too many requests, waiting 15 sec...")
            time.sleep(15)
            continue
        break

    result = response.json()
    time.sleep(1.5)   # be polite to the free API

    # empty result = this month is not published yet
    if response.status_code != 200 or not result.get("data"):
        return 0

    with open(file_path, "w") as f:
        json.dump(result, f, indent=1)
    return len(result["data"])

### Downloading US imports

In [4]:
us_missing = []
us_rows = 0
for period in months:
    rows = download_month(842, "M", period, US_FOLDER)
    if rows == 0:
        us_missing.append(period)
    us_rows += rows

print("US imports - files saved:", len(os.listdir(US_FOLDER)))
print("Total rows:", us_rows)
print("Months with no data yet:", us_missing)

US imports - files saved: 55
Total rows: 2056
Months with no data yet: ['202608']


### Downloading India exports

In [5]:
india_missing = []
india_rows = 0
for period in months:
    rows = download_month(699, "X", period, INDIA_FOLDER)
    if rows == 0:
        india_missing.append(period)
    india_rows += rows

print("India exports - files saved:", len(os.listdir(INDIA_FOLDER)))
print("Total rows:", india_rows)
print("Months with no data yet:", india_missing)

India exports - files saved: 52
Total rows: 2137
Months with no data yet: ['202201', '202202', '202203', '202608']


### Country code list

The API only gives numeric country codes (like 699 for India), not the names. Comtrade publishes a separate
reference file that maps codes to country names, so I downloaded that too.

In [6]:
codes_url = "https://comtradeapi.un.org/files/v1/app/reference/partnerAreas.json"
codes = requests.get(codes_url, timeout=60).json()

with open("../data/raw/partner_codes.json", "w") as f:
    json.dump(codes, f, indent=1)

print("Country codes saved:", len(codes["results"]))

Country codes saved: 310


### Quick look at one raw file

In [7]:
with open(f"{US_FOLDER}/2025-06.json") as f:
    sample = json.load(f)

print("Rows in June 2025:", sample["count"])
print("Columns per row:", len(sample["data"][0]))
sample["data"][1]

Rows in June 2025: 37
Columns per row: 47


{'typeCode': 'C',
 'freqCode': 'M',
 'refPeriodId': 20250601,
 'refYear': 2025,
 'refMonth': 6,
 'period': '202506',
 'reporterCode': 842,
 'reporterISO': None,
 'reporterDesc': None,
 'flowCode': 'M',
 'flowDesc': None,
 'partnerCode': 76,
 'partnerISO': None,
 'partnerDesc': None,
 'partner2Code': 0,
 'partner2ISO': None,
 'partner2Desc': None,
 'classificationCode': 'H6',
 'classificationSearchCode': 'HS',
 'isOriginalClassification': True,
 'cmdCode': '851713',
 'cmdDesc': None,
 'aggrLevel': None,
 'isLeaf': None,
 'customsCode': 'C00',
 'customsDesc': None,
 'mosCode': '0',
 'motCode': 0,
 'motDesc': None,
 'qtyUnitCode': 5,
 'qtyUnitAbbr': None,
 'qty': 4520.0,
 'isQtyEstimated': False,
 'altQtyUnitCode': 5,
 'altQtyUnitAbbr': None,
 'altQty': 4520.0,
 'isAltQtyEstimated': False,
 'netWgt': 330.187,
 'isNetWgtEstimated': True,
 'grossWgt': 0.0,
 'isGrossWgtEstimated': False,
 'cifvalue': 288698.0,
 'fobvalue': 284848.0,
 'primaryValue': 288698.0,
 'legacyEstimationFlag': 4,
 'is

Each row has **47 fields**, but many of them are empty (`None`) in the preview API, and the country is only a number.
That's what the cleaning notebook fixes. Next: `02_data_cleaning.ipynb`